# Introspecting a Datasource and a Model via MCP

SLayer's MCP (Model Context Protocol) server exposes tools like `query`, `models_summary`, `create_model`, and — the focus of this notebook — **`describe_datasource`** and **`inspect_model`**, which together tell an LLM agent everything it needs to know before issuing a query.

`inspect_model` returns a complete markdown view of a semantic model: metadata, all filters, dimensions, measures, custom aggregations, joins, every field reachable via joins (up to depth 5), and a sample-data table run against the live database. The Dimensions table gets a `sampled` column that carries the per-dim data profile inline — distinct values for string/boolean dims, `min .. max` for number/date/time dims. Every table auto-prunes columns that are empty for every row and collapses to a comma-separated backticked list when only one column survives.

It takes just two parameters:

| Parameter | Default | Effect |
|-----------|---------|--------|
| `model_name` | — required — | Name of the model to inspect |
| `num_rows` | `3` | Max rows in the sample-data table |

### A note on transport

`slayer mcp` runs over **stdio**, not HTTP, so a literal `curl` can't reach it. In this notebook we invoke each tool through the **in-process Python MCP client** — `await mcp.call_tool(name=..., arguments=...)` — which issues the same JSON-RPC `tools/call` request a desktop MCP client would issue over stdio. The REST API (`slayer serve` / port 5143) is a separate surface and is not used here.

We pick **`order_items`** as the subject of `inspect_model` because it has the richest join graph in Jaffle Shop: direct joins to `orders` and `products`, plus **multi-hop** joins to `customers` and `stores` reached via `orders`.

See also: [Joins](../05_joins/joins.md) · [Auto Ingestion](../03_auto_ingest) · [Multi-Stage Queries](../06_multistage_queries)

In [1]:
import json
import os
import sys

sys.path.insert(0, os.path.join(os.getcwd(), "..", "..", ".."))
sys.path.insert(0, os.path.join(os.getcwd(), "..", "jaffle_data"))

from setup_jaffle import ensure_jaffle_shop

engine, storage, models = ensure_jaffle_shop()
print(f"Loaded {len(models)} models: {sorted(m.name for m in models)}")

Loaded 8 models: ['customer_segments', 'customers', 'order_items', 'orders', 'products', 'stores', 'supplies', 'tweets']


## Why `order_items`?

During ingestion SLayer walks the FK graph and emits joins for every reachable table. `order_items` sits two hops away from `customers` and `stores`:

```
order_items ──► orders ──► customers
            │         └──► stores
            └──► products
```

Multi-hop joins are recorded with a **path-qualified** source column (e.g. `orders.customer_id`) so the generator knows which already-joined table to reach through. The next cell prints the raw `ModelJoin` entries so you can see the shape before we hand the model off to MCP.

In [2]:
order_items = next(m for m in models if m.name == "order_items")

print("order_items joins:")
for j in order_items.joins:
    src, tgt = j.join_pairs[0]
    hop = "multi-hop" if "." in src else "direct"
    print(f"  -> {j.target_model:<10} ON {src:<22} = {tgt:<5} ({hop})")

order_items joins:
  -> orders     ON order_id               = id    (direct)
  -> products   ON sku                    = sku   (direct)
  -> customers  ON orders.customer_id     = id    (multi-hop)
  -> stores     ON orders.store_id        = id    (multi-hop)


## Wiring up the MCP client

We instantiate the **MCP** server (from `slayer.mcp.server`, _not_ the REST app in `slayer.api.server`) against the same storage the engine just loaded. `create_mcp_server` returns a `FastMCP` instance; its `list_tools()` and `call_tool(name, arguments)` methods are exactly what stdio clients invoke under the hood.

Over stdio, each `call_tool` request serialises to a JSON-RPC frame like this (the `params` payload is what would be the POST body if SLayer's MCP ran on HTTP):

```json
{
  "jsonrpc": "2.0", "id": 1, "method": "tools/call",
  "params": {
    "name": "inspect_model",
    "arguments": {"model_name": "order_items", "num_rows": 10}
  }
}
```

In [3]:
from slayer.mcp.server import create_mcp_server

mcp = create_mcp_server(storage=storage)

tools = await mcp.list_tools()
print(f"MCP server exposes {len(tools)} tools:")
for t in tools:
    marker = "<-- targets" if t.name in ("describe_datasource", "inspect_model") else ""
    print(f"  - {t.name} {marker}")

inspect_tool = next(t for t in tools if t.name == "inspect_model")
print("\ninspect_model input schema:")
print(json.dumps(inspect_tool.inputSchema, indent=2))

MCP server exposes 13 tools:
  - help 
  - query 
  - models_summary 
  - inspect_model <-- targets
  - create_model 
  - edit_model 
  - create_datasource 
  - list_datasources 
  - describe_datasource <-- targets
  - edit_datasource 
  - delete_model 
  - delete_datasource 
  - ingest_datasource_models 

inspect_model input schema:
{
  "properties": {
    "model_name": {
      "title": "Model Name",
      "type": "string"
    },
    "num_rows": {
      "default": 3,
      "title": "Num Rows",
      "type": "integer"
    }
  },
  "required": [
    "model_name"
  ],
  "title": "inspect_modelArguments",
  "type": "object"
}


## Step 1 — Introspect the datasource

Before drilling into a specific model, a well-behaved agent verifies the datasource: is it reachable, what's its type, what schemas are available, and what tables live there? The MCP `describe_datasource` tool covers all of that in one call. Pass `list_tables=False` to skip the table-listing block if you just want the connection probe; pass `schema_name="..."` to scope the table listing.

The Jaffle Shop fixture registered a DuckDB datasource named `jaffle_shop` when we called `ensure_jaffle_shop()`, so let's inspect it.

In [4]:
describe_args = {"name": "jaffle_shop"}
print("Arguments (the 'curl POST body' equivalent):")
print(json.dumps(describe_args, indent=2))

content, _ = await mcp.call_tool(name="describe_datasource", arguments=describe_args)
print("\n=== describe_datasource response ===\n")
print(content[0].text)

Arguments (the 'curl POST body' equivalent):
{
  "name": "jaffle_shop"
}



=== describe_datasource response ===

Datasource: jaffle_shop
Type: duckdb
Database: /home/james/Dropbox/SLayer/slayer/docs/examples/jaffle_data/jaffle_shop.duckdb

Connection: OK
Available schemas: jaffle_shop.main, system.information_schema, system.main, temp.main

Tables (8):
  - customer_segments
  - customers
  - order_items
  - orders
  - products
  - stores
  - supplies
  - tweets

Use ingest_datasource_models to create models from these tables.


## Step 2 — Survey the datasource's models with `models_summary`

Now that we know the datasource works, `models_summary` gives a compact markdown overview of every model registered against it: name, description, a `name`+`description` table of its dimensions and measures, and the list of models it joins to. It's deliberately lighter than `inspect_model` — no types, no distinct values, no sample data — so an agent can scan N models without making N heavy calls.

In [5]:
summary_args = {"datasource_name": "jaffle_shop"}
print("Arguments (the 'curl POST body' equivalent):")
print(json.dumps(summary_args, indent=2))

content, _ = await mcp.call_tool(name="models_summary", arguments=summary_args)
print("\n=== models_summary response (markdown) ===\n")
print(content[0].text)

Arguments (the 'curl POST body' equivalent):
{
  "datasource_name": "jaffle_shop"
}



=== models_summary response (markdown) ===

# Datasource: `jaffle_shop` — 8 model(s)

## `customer_segments`
**Dimensions (2):**

`id`, `segment`

**Measures (1):**

`segment`

**Joins to:** _(none)_

## `customers`
**Dimensions (2):**

`id`, `name`

**Measures (1):**

`name`

**Joins to:** _(none)_

## `order_items`
**Dimensions (4):**

`id`, `order_id`, `sku`, `quantity`

**Measures (2):**

`quantity`, `sku`

**Joins to:** `customers`, `orders`, `products`, `stores`

## `orders`
**Dimensions (4):**

`id`, `customer_id`, `ordered_at`, `store_id`

**Measures (4):**

`subtotal`, `tax_paid`, `order_total`, `ordered_at`

**Joins to:** `customers`, `stores`

## `products`
**Dimensions (4):**

`sku`, `name`, `type`, `description`

**Measures (4):**

`price`, `name`, `type`, `description`

**Joins to:** _(none)_

## `stores`
**Dimensions (3):**

`id`, `name`, `opened_at`

**Measures (3):**

`tax_rate`, `name`, `opened_at`

**Joins to:** _(none)_

## `supplies`
**Dimensions (4):**

`id`, `na

## Step 3 — Drill into one model with `inspect_model`

We've picked `order_items` from the `models_summary` output above. `inspect_model` returns a single markdown document — we print it verbatim rather than parsing as JSON.

In [6]:
arguments = {"model_name": "order_items", "num_rows": 10}
print("Arguments (the 'curl POST body' equivalent):")
print(json.dumps(arguments, indent=2))

content, _ = await mcp.call_tool(name="inspect_model", arguments=arguments)
markdown = content[0].text

print(f"\nResponse length: {len(markdown):,} characters")
print("\n=== inspect_model response (markdown) ===\n")
print(markdown)

Arguments (the 'curl POST body' equivalent):
{
  "model_name": "order_items",
  "num_rows": 10
}



Response length: 2,135 characters

=== inspect_model response (markdown) ===

# Model: `order_items`

- **data_source:** `jaffle_shop`
- **sql_table:** `order_items`
- **row_count:** 324,415

## Dimensions (4)

| name | type | primary_key | sql | sampled |
| --- | --- | --- | --- | --- |
| id | string | yes | id | — |
| order_id | string | — | order_id | > 20 distinct |
| sku | string | — | sku | `JAF-005`, `JAF-001`, `BEV-001`, `BEV-005`, `JAF-003`, `BEV-003`, `JAF-002`, `BEV-002`, `JAF-004`, `BEV-004` |
| quantity | number | — | quantity | 1 .. 5 |

## Measures (2)

| name | sql | allowed_aggregations |
| --- | --- | --- |
| quantity | quantity | all |
| sku | sku | all |

## Joins (4)

| target_model | join_pairs | kind |
| --- | --- | --- |
| orders | order_id = id | direct |
| products | sku = sku | direct |
| customers | orders.customer_id = id | multi-hop |
| stores | orders.store_id = id | multi-hop |

## Reachable via joins (max depth: 5)

**Dimensions (9):** `orders.customer

## What we just got

The exploration flow was `describe_datasource` → `models_summary` → `inspect_model` — exactly the sequence an MCP-driven agent would run before composing its first `query` call.

`describe_datasource` gave us the datasource type (`duckdb`), the file backing it, a successful connection probe, the available schemas, and the list of tables in one call. `models_summary` returned a brief overview of every model in that datasource. `inspect_model` returned a single markdown document covering:

- `# Model: <name>` header and description
- Metadata bullets: `data_source`, `sql_table`, `default_time_dimension`, `row_count`
- `## SQL` fenced block when the model has a custom SQL expression (query-backed models)
- `## Filters (model-level)` — always-applied WHERE conditions
- `## Dimensions` — markdown table including a `sampled` column that embeds the per-dim data profile: distinct values (e.g. the SKUs for `sku`) for string/boolean dims, and `<min> .. <max>` for number/date/time dims
- `## Measures` — markdown table with `name`, `sql`, `allowed_aggregations`, `filter`, `label`, `description`
- `## Aggregations` — custom `Aggregation` definitions with their formulas
- `## Joins` — target model, join pairs, and `direct`/`multi-hop` indicator
- `## Reachable via joins (max depth: 5)` — every fully-qualified dimension and measure path reachable through the join graph (so an LLM doesn't need to call `inspect_model` for each joined model)
- `## Sample Data` — a real `SlayerQuery` executed through `engine.execute`, with `*:count` plus one aggregation per measure (`avg` when allowed, otherwise the first permitted aggregation)

Every markdown table in the response auto-prunes empty columns. When exactly one column survives pruning, the table collapses to a comma-separated backticked list — which is why a dim or measure table from `models_summary` on an auto-ingested model (where descriptions are often absent) often renders as a dense list of names rather than a two-column table of mostly em-dashes.

Other MCP tools that complement these for an exploring agent: `list_datasources` (enumerate datasources) and `models_summary(datasource_name="...")` — the latter returns a compact markdown summary of every model in one datasource (name + description + dim/measure name tables + list of joined model names), much lighter than `inspect_model` and great for picking the right model to drill into next.